# Debug: NOAA Degree Days (`fetch_noaa.py`)

Step-by-step debug notebook for the NOAA Climate Data Online (CDO) pipeline.

**Pipeline:** `pipeline/fetch_noaa.py`  
**Output:** `data/noaa_degree_days.parquet`  
**Source:** NOAA CDO API v2 — requires `NOAA_CDO_TOKEN`

---
Set your token in a `.env` file or export `NOAA_CDO_TOKEN` before running.  
Register for a free token (instant) at https://www.ncdc.noaa.gov/cdo-web/token

## 1. Imports & Dependencies

In [ ]:
import importlib
import os
import sys
import time
from datetime import date, timedelta
from pathlib import Path

required = ["pandas", "requests", "pyarrow", "dotenv"]
missing = []
for pkg in required:
    if importlib.util.find_spec(pkg) is None:
        missing.append(pkg)

if missing:
    print(f"FAIL  Missing packages: {missing}")
else:
    import pandas as pd
    import requests
    from dotenv import load_dotenv
    load_dotenv()
    print(f"PASS  All packages available — pandas {pd.__version__}")

## 2. API Token Check

In [ ]:
TOKEN = os.environ.get("NOAA_CDO_TOKEN", "")

if not TOKEN:
    print("FAIL  NOAA_CDO_TOKEN is not set")
    print("      Options:")
    print("        1. Create a .env file in the project root with: NOAA_CDO_TOKEN=your_token")
    print("        2. Export in shell: export NOAA_CDO_TOKEN=your_token")
    print("        3. Register free (instant) at https://www.ncdc.noaa.gov/cdo-web/token")
else:
    masked = TOKEN[:4] + "*" * max(0, len(TOKEN) - 4)
    print(f"PASS  NOAA_CDO_TOKEN found: {masked}")

## 3. Configuration & Station List

In [ ]:
CDO_BASE_URL = "https://www.ncdc.noaa.gov/cdo-web/api/v2"
CDO_PAGE_SIZE = 1000
ROLLING_YEARS = 5
OUTPUT_PATH = Path("../data/noaa_degree_days.parquet")

STATIONS = [
    ("GHCND:USW00094728", "New York City (Central Park), NY"),
    ("GHCND:USW00014742", "Boston Logan, MA"),
    ("GHCND:USW00094847", "Chicago O'Hare, IL"),
    ("GHCND:USW00014739", "Detroit Metro, MI"),
    ("GHCND:USW00013880", "Atlanta Hartsfield, GA"),
    ("GHCND:USW00012960", "Houston Intercontinental, TX"),
    ("GHCND:USW00014918", "Minneapolis-St. Paul, MN"),
    ("GHCND:USW00023174", "Los Angeles, CA"),
    ("GHCND:USW00093721", "Denver, CO"),
    ("GHCND:USW00023234", "Seattle-Tacoma, WA"),
    ("GHCND:USW00013958", "Dallas/Fort Worth, TX"),
    ("GHCND:USW00013994", "Miami, FL"),
]
STATION_IDS = [s[0] for s in STATIONS]

end_date = date.today()
start_date = end_date - timedelta(days=ROLLING_YEARS * 365)

print(f"Stations   : {len(STATIONS)}")
print(f"Date range : {start_date} → {end_date}")
print(f"Output     : {OUTPUT_PATH.resolve()}")
print()
print("Station list:")
for sid, name in STATIONS:
    print(f"  {sid}  {name}")

## 4. Connectivity Test — Single Station, Short Window

Fetch 7 days of data from NYC (Central Park) to verify the CDO API is reachable.

In [ ]:
if not TOKEN:
    print("SKIP  No token — set NOAA_CDO_TOKEN and re-run")
else:
    test_station = STATION_IDS[0]  # NYC Central Park
    test_end = end_date.isoformat()
    test_start = (end_date - timedelta(days=14)).isoformat()

    url = f"{CDO_BASE_URL}/data"
    params = {
        "datasetid": "GHCND",
        "stationid": test_station,
        "datatypeid": "TMAX,TMIN",
        "startdate": test_start,
        "enddate": test_end,
        "units": "standard",
        "limit": 100,
        "offset": 1,
    }
    headers = {"token": TOKEN}

    print(f"Testing: {test_station} from {test_start} to {test_end}")
    print(f"URL    : {url}")
    print()

    try:
        resp = requests.get(url, headers=headers, params=params, timeout=30)
        print(f"Status : {resp.status_code}")

        if resp.status_code == 400:
            print("FAIL  400 Bad Request — check station ID or date range")
            print(f"      Response: {resp.text[:300]}")
        elif resp.status_code == 401:
            print("FAIL  401 Unauthorized — NOAA_CDO_TOKEN is invalid")
        elif resp.status_code != 200:
            print(f"FAIL  Unexpected status {resp.status_code}: {resp.text[:300]}")
        else:
            payload = resp.json()
            results = payload.get("results", [])
            meta = payload.get("metadata", {}).get("resultset", {})
            print(f"PASS  {len(results)} records returned (total in window: {meta.get('count', '?')})")
            print()
            print("Sample records:")
            for r in results[:6]:
                print(f"  {r}")
    except requests.Timeout:
        print("FAIL  Timed out — check network")
    except Exception as e:
        print(f"FAIL  Exception: {e}")

## 5. Inspect Pagination

The CDO API returns at most 1000 records per call. Check how many pages a 5-year window for one station requires.

In [ ]:
if not TOKEN:
    print("SKIP")
elif resp.status_code != 200:
    print("SKIP — fix API connection first")
else:
    # Estimate record count for a full 5-year window
    # Each station: 2 datatypes (TMAX, TMIN) × 365 days × 5 years ≈ 3650 records
    params_count = {
        "datasetid": "GHCND",
        "stationid": STATION_IDS[0],
        "datatypeid": "TMAX,TMIN",
        "startdate": start_date.isoformat(),
        "enddate": end_date.isoformat(),
        "units": "standard",
        "limit": 1,
        "offset": 1,
    }
    r = requests.get(url, headers=headers, params=params_count, timeout=30)
    if r.status_code == 200:
        meta = r.json().get("metadata", {}).get("resultset", {})
        total = int(meta.get("count", 0))
        pages = -(-total // CDO_PAGE_SIZE)  # ceiling division
        print(f"Station {STATION_IDS[0]}:")
        print(f"  Total records in 5yr window : {total}")
        print(f"  Pages needed (at {CDO_PAGE_SIZE}/page): {pages}")
        print(f"  Estimated API calls for all {len(STATIONS)} stations: {pages * len(STATIONS)}")
    else:
        print(f"Could not get count: {r.status_code}")

## 6. Fetch All Stations (Full Pipeline)

This mirrors the production pipeline. Expect ~1–3 minutes due to CDO rate limits (0.25s delay between pages).

In [ ]:
def fetch_cdo_data(token, station_id, datatype_ids, start, end):
    headers = {"token": token}
    base_params = {
        "datasetid": "GHCND",
        "stationid": station_id,
        "datatypeid": ",".join(datatype_ids),
        "startdate": start,
        "enddate": end,
        "units": "standard",
        "limit": CDO_PAGE_SIZE,
    }
    all_rows = []
    offset = 1
    while True:
        params = {**base_params, "offset": offset}
        r = requests.get(f"{CDO_BASE_URL}/data", headers=headers,
                         params=params, timeout=30)
        if r.status_code == 400:
            break
        if r.status_code != 200:
            print(f"  WARNING: {r.status_code} for {station_id} offset={offset}")
            break
        payload = r.json()
        results = payload.get("results", [])
        if not results:
            break
        all_rows.extend(results)
        meta = payload.get("metadata", {}).get("resultset", {})
        total = int(meta.get("count", 0))
        if offset + CDO_PAGE_SIZE > total:
            break
        offset += CDO_PAGE_SIZE
        time.sleep(0.25)
    if not all_rows:
        return pd.DataFrame()
    df = pd.DataFrame(all_rows)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df["station"] = station_id
    return df[["date", "datatype", "value", "station"]]


if not TOKEN:
    print("SKIP  No token")
else:
    frames = []
    station_results = {}
    start_str = start_date.isoformat()
    end_str = end_date.isoformat()

    for sid, name in STATIONS:
        print(f"  Fetching {sid} ({name}) ...", end=" ", flush=True)
        df = fetch_cdo_data(TOKEN, sid, ["TMAX", "TMIN"], start_str, end_str)
        if df.empty:
            print("no data (SKIP)")
            station_results[sid] = 0
        else:
            print(f"{len(df)} records")
            station_results[sid] = len(df)
            frames.append(df)

    print()
    print(f"PASS  {len(frames)}/{len(STATIONS)} stations returned data")

## 7. HDD / CDD Computation

In [ ]:
if not TOKEN or not frames:
    print("SKIP")
else:
    all_temps = pd.concat(frames, ignore_index=True)
    print(f"Combined raw records : {len(all_temps)}")
    print(f"Stations with data   : {all_temps['station'].nunique()}")
    print(f"Datatypes            : {sorted(all_temps['datatype'].unique())}")
    print()

    # Check for missing TMAX or TMIN
    tmax_count = (all_temps["datatype"] == "TMAX").sum()
    tmin_count = (all_temps["datatype"] == "TMIN").sum()
    print(f"TMAX records: {tmax_count}")
    print(f"TMIN records: {tmin_count}")
    if abs(tmax_count - tmin_count) > tmax_count * 0.05:
        print("WARN  TMAX/TMIN counts differ by more than 5% — some days may be missing one type")
    else:
        print("PASS  TMAX/TMIN counts are balanced")
    print()

    # Pivot and compute HDD/CDD
    pivot = all_temps.pivot_table(
        index=["date", "station"], columns="datatype",
        values="value", aggfunc="mean"
    ).reset_index()

    before = len(pivot)
    pivot = pivot.dropna(subset=["TMAX", "TMIN"])
    after = len(pivot)
    dropped = before - after
    if dropped > 0:
        print(f"WARN  Dropped {dropped} pivot rows missing TMAX or TMIN")

    pivot["tavg"] = (pivot["TMAX"] + pivot["TMIN"]) / 2.0
    pivot["hdd"] = (65.0 - pivot["tavg"]).clip(lower=0.0)
    pivot["cdd"] = (pivot["tavg"] - 65.0).clip(lower=0.0)

    daily = pivot[["date", "station", "hdd", "cdd"]]

    print("Daily HDD/CDD stats (across all stations):")
    print(daily[["hdd", "cdd"]].describe().round(2).to_string())
    print()

    # Sanity check: HDD and CDD should never both be positive on the same row
    both_positive = daily[(daily["hdd"] > 0) & (daily["cdd"] > 0)]
    if not both_positive.empty:
        print(f"FAIL  {len(both_positive)} rows where both HDD>0 and CDD>0 (impossible!)")
    else:
        print("PASS  No rows with both HDD>0 and CDD>0")

## 8. Weekly Aggregation

In [ ]:
if not TOKEN or not frames:
    print("SKIP")
else:
    # Average across stations per day
    daily_avg = daily.groupby("date")[["hdd", "cdd"]].mean().reset_index()

    # Assign ISO week start (Monday)
    daily_avg["week_start"] = daily_avg["date"] - pd.to_timedelta(
        daily_avg["date"].dt.dayofweek, unit="D"
    )

    weekly = (
        daily_avg.groupby("week_start")[["hdd", "cdd"]]
        .sum()
        .rename(columns={"hdd": "hdd_weekly", "cdd": "cdd_weekly"})
        .reset_index()
        .sort_values("week_start")
        .reset_index(drop=True)
    )

    print(f"Weekly rows: {len(weekly)}")
    print(f"Date range : {weekly['week_start'].min().date()} → {weekly['week_start'].max().date()}")
    print()
    print("Weekly HDD/CDD stats:")
    print(weekly[["hdd_weekly", "cdd_weekly"]].describe().round(1).to_string())
    print()
    print("Latest 5 weeks:")
    print(weekly.tail(5).to_string())

    # Gap detection in weekly data
    gaps = weekly["week_start"].diff().dt.days
    large_gaps = gaps[gaps > 14]
    if not large_gaps.empty:
        print(f"\nWARN  {len(large_gaps)} weekly gaps > 14 days:")
        for idx in large_gaps.index:
            print(f"  {weekly.loc[idx-1, 'week_start'].date()} → {weekly.loc[idx, 'week_start'].date()} ({gaps[idx]:.0f}d)")
    else:
        print("\nPASS  No large gaps in weekly data")

## 9. Station Coverage Diagnostics

In [ ]:
if not TOKEN or not frames:
    print("SKIP")
else:
    print("Record counts per station:")
    station_summary = (
        daily.groupby("station").agg(
            days=("date", "count"),
            first=("date", "min"),
            last=("date", "max"),
            avg_hdd=("hdd", "mean"),
            avg_cdd=("cdd", "mean"),
        )
    )
    # Add city name
    name_map = {sid: name for sid, name in STATIONS}
    station_summary["city"] = station_summary.index.map(name_map)
    print(station_summary[["city", "days", "first", "last", "avg_hdd", "avg_cdd"]].to_string())

    expected_days = ROLLING_YEARS * 365
    sparse = station_summary[station_summary["days"] < expected_days * 0.8]
    if not sparse.empty:
        print(f"\nWARN  {len(sparse)} station(s) have < 80% expected coverage:")
        print(sparse[["city", "days"]].to_string())
    else:
        print("\nPASS  All stations meet coverage threshold")

## 10. Compare to Saved Parquet

In [ ]:
if OUTPUT_PATH.exists():
    saved = pd.read_parquet(OUTPUT_PATH)
    print(f"Saved parquet: {len(saved)} rows")
    print(f"Columns      : {list(saved.columns)}")
    print(f"Date range   : {saved['week_start'].min().date()} → {saved['week_start'].max().date()}")
    print()
    print("Latest 5 saved rows:")
    print(saved.tail(5).to_string())

    if TOKEN and frames:
        fresh_max = weekly["week_start"].max()
        saved_max = saved["week_start"].max()
        lag = (fresh_max - saved_max).days
        if lag > 14:
            print(f"\nWARN  Saved parquet is {lag} days behind fresh fetch")
        else:
            print(f"\nPASS  Saved parquet is up to date (lag = {lag} days)")
else:
    print(f"INFO  No saved parquet at {OUTPUT_PATH} — run pipeline to generate it")

## 11. Write Output (optional)

In [ ]:
# if TOKEN and frames:
#     OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
#     weekly.to_parquet(OUTPUT_PATH, index=False)
#     print(f"Wrote {len(weekly)} rows → {OUTPUT_PATH}")